In [0]:
import pandas as pd

df = pd.read_csv("/Workspace/Users/kushal.vadukul2006@gmail.com/Multi-Class-Credit-Card-Risk-Model/UCI_Credit_Card.csv")

df

In [0]:

df1 = df.copy()

def risk(x):
    if x <= 0:
        return 0
    elif x == 1:
        return 1
    elif x == 2:
        return 2
    else:
        return 3
        

df1['Credit_Risk'] = df['PAY_0'].apply(risk)

df1



In [0]:
%pip install scikit-optimize
%pip install xgboost
%pip install pytorch-tabular
%pip install optuna

In [0]:
from sklearn.model_selection import train_test_split

df1['maxedout_percent'] = (df1['BILL_AMT1']) / (df1['LIMIT_BAL'] +1)
df1['paid_percent'] = df1['PAY_AMT1'] / (df1['BILL_AMT1'].abs() + 1)
df1['pay_increase'] = (df1['PAY_AMT1'] - df1['PAY_AMT6']) / (df1['PAY_AMT6'] + 1)

features = ['PAY_2','PAY_3','PAY_4','PAY_5','PAY_6', 'BILL_AMT1','BILL_AMT2','BILL_AMT3','BILL_AMT4','BILL_AMT5','BILL_AMT6','PAY_AMT1','PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5','PAY_AMT6','EDUCATION']

target = 'Credit_Risk'


X_train, X_test, y_train, y_test = train_test_split(df1[features], df1[target], test_size=0.2, random_state=42)

train_df1 = X_train.copy()
train_df1[target] = y_train
education_risk = train_df1.groupby('EDUCATION')[target].mean()
df1['education_risk'] = df1['EDUCATION'].map(education_risk)



features = ['PAY_2','PAY_3','PAY_4','PAY_5','PAY_6', 'BILL_AMT1','BILL_AMT2','BILL_AMT3','BILL_AMT4','BILL_AMT5','BILL_AMT6','PAY_AMT1','PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5','PAY_AMT6','EDUCATION','education_risk']

X_train, X_test, y_train, y_test = train_test_split(df1[features], df1[target], test_size=0.2, random_state=42)

In [0]:
import xgboost as xgb
from sklearn.metrics import classification_report, confusion_matrix, f1_score

Model = xgb.XGBClassifier(n_estimators= 500, objective='multi:softmax', num_class=4, random_state=42,reg_alpha= 0.7322710337614933, reg_lambda= 13.072035773567185, learning_rate= 0.10670381684729559, max_depth= 3, min_child_weight= 9, colsample_bytree= 0.4067336093171891, subsample= 0.6379121527799684)

Model.fit(X_train, y_train)

y_prediction = Model.predict(X_test)

print(classification_report(y_test, y_prediction))
print(confusion_matrix(y_test, y_prediction))
macro_f1 = f1_score(y_test, y_prediction, average='macro')
print(f"Macro F1-Score: {macro_f1:.4f}")






In [0]:
%skip
import optuna
from xgboost import XGBClassifier
from sklearn.metrics import f1_score
def objective(trial):
    
    params = {
        'reg_alpha': trial.suggest_float('reg_alpha', 0.1, 1.5, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 10.0, 30.0, log=True),
        'learning_rate': trial.suggest_float('learning_rate', 0.10, 0.18, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 4),        
        'min_child_weight': trial.suggest_int('min_child_weight', 5, 9), 
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 0.7),
        'subsample': trial.suggest_float('subsample', 0.6, 0.8),
        'n_estimators': 500, 
        'objective': 'multi:softmax',
        'num_class': 4,
        'random_state': 42
    }
    
  
    model = XGBClassifier(**params)
    model.fit(X_train, y_train)
    
    
    preds = model.predict(X_test)
    macro_f1 = f1_score(y_test, preds, average='macro')
    
    return macro_f1

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)  


print("Best XGBoost Macro F1 Score:", study.best_value)
print("Best XGBoost Hyperparameters:", study.best_params)

In [0]:
from pytorch_tabular import TabularModel
from pytorch_tabular.config import DataConfig, ModelConfig, TrainerConfig, OptimizerConfig
from pytorch_tabular.models import FTTransformerConfig
from sklearn.metrics import f1_score
from pytorch_tabular.utils import get_balanced_sampler
import optuna


train_df2 = X_train.copy()
train_df2['Credit_Risk'] = y_train
test_df2 = X_test.copy()
test_df2['Credit_Risk'] = y_test



data = DataConfig(
    target=['Credit_Risk'],
    continuous_cols=['PAY_2','PAY_3','PAY_4','PAY_5','PAY_6', 'BILL_AMT1','BILL_AMT2','BILL_AMT3','BILL_AMT4','BILL_AMT5','BILL_AMT6','PAY_AMT1','PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5','PAY_AMT6','education_risk'],
    categorical_cols=['EDUCATION'],
)

trainer = TrainerConfig(batch_size=64, max_epochs=10, accelerator="auto", early_stopping=None)

optimizer = OptimizerConfig()

model = FTTransformerConfig(
    task="classification",
    input_embed_dim=16,
    num_attn_blocks=5,
    num_heads=8,
    learning_rate=0.0007554636976519865,
    attn_dropout=0.11042811999287684,
    ff_dropout=0.11042811999287684,
    add_norm_dropout=0.11042811999287684,
    embedding_dropout=0.11042811999287684,
)

tabular_model = TabularModel(
    data_config=data,
    model_config=model,
    optimizer_config=optimizer,
    trainer_config=trainer
)

train_sampler = get_balanced_sampler(train_df2['Credit_Risk'])

tabular_model.fit(train=train_df2, validation=test_df2, train_sampler=train_sampler)

result = tabular_model.evaluate(test=test_df2)
print(result)

preds_df = tabular_model.predict(test_df2)
y_pred = preds_df['Credit_Risk_prediction']
FTmacro_f1 = f1_score(test_df2['Credit_Risk'], y_pred, average='macro')
print(f"Macro F1-Score: {FTmacro_f1:.4f}")



In [0]:
from pytorch_tabular import TabularModel
from pytorch_tabular.config import DataConfig, ModelConfig, TrainerConfig, OptimizerConfig
from pytorch_tabular.models import FTTransformerConfig
from sklearn.metrics import f1_score
from pytorch_tabular.utils import get_balanced_sampler
import optuna


train_df2 = X_train.copy()
train_df2['Credit_Risk'] = y_train
test_df2 = X_test.copy()
test_df2['Credit_Risk'] = y_test

def objective(trial):
    lr = trial.suggest_float("learning_rate", 1e-4, 5e-3, log=True)
    input_embed_dim = trial.suggest_categorical("input_embed_dim", [16, 32, 64])
    num_attn_blocks = trial.suggest_int("num_attn_blocks", 2, 5)
    num_heads = trial.suggest_categorical("num_heads", [2, 4, 8])
    dropout_val = trial.suggest_float("dropout", 0.05, 0.3)

    data = DataConfig(
        target=['Credit_Risk'],
        continuous_cols=['PAY_2','PAY_3','PAY_4','PAY_5','PAY_6', 'BILL_AMT1','BILL_AMT2','BILL_AMT3','BILL_AMT4','BILL_AMT5','BILL_AMT6','PAY_AMT1','PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5','PAY_AMT6','education_risk'],
        categorical_cols=['EDUCATION'],
    )

    trainer = TrainerConfig(batch_size=128, max_epochs=7, accelerator="auto", early_stopping=None)

    optimizer = OptimizerConfig()

    model = FTTransformerConfig(
        task="classification",
        input_embed_dim=input_embed_dim,
        num_attn_blocks=num_attn_blocks,
        num_heads=num_heads,
        learning_rate=lr,
        attn_dropout=dropout_val,
        ff_dropout=dropout_val,
        add_norm_dropout=dropout_val,
        embedding_dropout=dropout_val,
    )

    tabular_model = TabularModel(
        data_config=data,
        model_config=model,
        optimizer_config=optimizer,
        trainer_config=trainer
    )

    train_sampler = get_balanced_sampler(train_df2['Credit_Risk'])

    tabular_model.fit(train=train_df2, validation=test_df2, train_sampler=train_sampler)

    result = tabular_model.evaluate(test=test_df2)
    print(result)

    preds_df = tabular_model.predict(test_df2)
    y_pred = preds_df['Credit_Risk_prediction']
    true_macro_f1 = f1_score(test_df2['Credit_Risk'], y_pred, average='macro')

    return true_macro_f1

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=8)


print("Best Macro F1 Score:", study.best_value)
print("Best Hyperparameters:", study.best_params)

In [0]:


preds_df